In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [28]:
# Quantum random number generator

simulator = BasicSimulator()

def quantum_random() -> int:
    # create a circuit (1 qubit, 1 classical bit)
    qc = QuantumCircuit(1, 1)
    # apply hadamard gate
    qc.h(0)
    # measure qubit into classical bit
    qc.measure(0, 0)
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1) # run the circuit once
    result = job.result()
    return int(list(result.get_counts().keys())[0])

def quantum_random_bits(n: int) -> list:
  # generate n quantum-random bits
  return [quantum_random() for _ in range(n)]

sample = quantum_random_bits(10)
print("Sample quantum random bits:", sample)

Sample quantum random bits: [1, 1, 1, 0, 0, 1, 1, 1, 1, 0]


In [29]:
# Preparing Qubits (Alice)
def prepare_qubit(bit: int, basis: int) -> QuantumCircuit:
  # create 1-qubit circuit
  qc = QuantumCircuit(1)

  # apply X gate to flip |0⟩ → |1⟩
  if bit == 1:
    qc.x(0)

  # if using diagonal basis, apply hadamard gate
  if basis == 1:
    qc.h(0)

  return qc

In [30]:
# Measuring Qubits (Bob)
def measure_qubit(state_qc: QuantumCircuit, basis: int) -> int:
  # copy the incoming qubit circuit so we don't modify original
  qc = state_qc.copy()

  # add a classical register to store measurement result
  qc.add_register(__import__('qiskit').ClassicalRegister(1))

  # if diagonal basis, rotate using hadamard before measuring
  if basis == 1:
    qc.h(0)

  # measure qubit into classical bit
  qc.measure(0, 0)

  # run measurement once
  job = simulator.run(transpile(qc, simulator), shots=1)
  result = job.result()
  return int(list(result.get_counts().keys())[0])

In [31]:
# Intercept-Resend Attack (Eve)
def intercept_resend(state_qc: QuantumCircuit) -> tuple:
  basis = quantum_random()
  bit = measure_qubit(state_qc, basis)
  forwarded = prepare_qubit(bit, basis)
  return forwarded, bit, basis

In [32]:
# Full BB84 Simulation with configurable attacker
NUM_QUBITS = 40
EVE_INTERCEPT = 1.0
SAMPLE_FRACTION =  0.5
ATTACK_THRESHOLD = 0.1

# ALICE
alice_qubits = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

transmitted_qubits = [prepare_qubit(alice_qubits[i], alice_bases[i]) for i in range(NUM_QUBITS)]

# EVE
eve_bases = [None] * NUM_QUBITS
eve_bits = [None] * NUM_QUBITS
qubits_for_bob = []

intercept_decisions = quantum_random_bits(NUM_QUBITS)

for i in range(NUM_QUBITS):
    # Eve intercepts this qubit with probability EVE_INTERCEPT
    intercept = (intercept_decisions[i] == 0) if EVE_INTERCEPT < 1.0 else True
    if intercept:
        forwarded, eb, ebit = intercept_resend(transmitted_qubits[i])
        qubits_for_bob.append(forwarded)
        eve_bases[i] = eb
        eve_bits[i]  = ebit
    else:
        qubits_for_bob.append(transmitted_qubits[i])

# BOB
bob_bases = quantum_random_bits(NUM_QUBITS)
bob_measurements = [measure_qubit(qubits_for_bob[i], bob_bases[i]) for i in range(NUM_QUBITS)]

In [33]:
# Print raw transmission table
print("Raw Transmission")
for i in range(NUM_QUBITS):
    intercepted = "YES" if eve_bases[i] is not None else "no"
    print(f"  [{i:2d}] Alice: bit={alice_qubits[i]} basis={alice_bases[i]} | "
          f"Eve: {intercepted:3s} | "
          f"Bob: basis={bob_bases[i]} bit={bob_measurements[i]}")

Raw Transmission
  [ 0] Alice: bit=0 basis=1 | Eve: YES | Bob: basis=1 bit=1
  [ 1] Alice: bit=0 basis=0 | Eve: YES | Bob: basis=0 bit=0
  [ 2] Alice: bit=0 basis=1 | Eve: YES | Bob: basis=1 bit=0
  [ 3] Alice: bit=0 basis=0 | Eve: YES | Bob: basis=1 bit=1
  [ 4] Alice: bit=0 basis=0 | Eve: YES | Bob: basis=1 bit=0
  [ 5] Alice: bit=1 basis=1 | Eve: YES | Bob: basis=0 bit=0
  [ 6] Alice: bit=1 basis=0 | Eve: YES | Bob: basis=0 bit=1
  [ 7] Alice: bit=1 basis=0 | Eve: YES | Bob: basis=0 bit=1
  [ 8] Alice: bit=1 basis=0 | Eve: YES | Bob: basis=1 bit=0
  [ 9] Alice: bit=0 basis=1 | Eve: YES | Bob: basis=0 bit=1
  [10] Alice: bit=1 basis=1 | Eve: YES | Bob: basis=1 bit=0
  [11] Alice: bit=1 basis=1 | Eve: YES | Bob: basis=0 bit=1
  [12] Alice: bit=0 basis=1 | Eve: YES | Bob: basis=1 bit=0
  [13] Alice: bit=0 basis=0 | Eve: YES | Bob: basis=0 bit=0
  [14] Alice: bit=0 basis=1 | Eve: YES | Bob: basis=1 bit=1
  [15] Alice: bit=1 basis=1 | Eve: YES | Bob: basis=1 bit=0
  [16] Alice: bit=0 bas

In [34]:
# Basis Reconciliation
matching_indices = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_qubits[i] for i in matching_indices]
bob_key   = [bob_measurements[i] for i in matching_indices]

print("Sifted Key")
print(f"Matching positions: {matching_indices}")
print(f"Alice's sifted key: {alice_key}")
print(f"Bob's sifted key:   {bob_key}")
errors_in_sifted = sum(a != b for a, b in zip(alice_key, bob_key))
print(f"Errors in sifted key: {errors_in_sifted}/{len(alice_key)}")

Sifted Key
Matching positions: [0, 1, 2, 6, 7, 10, 12, 13, 14, 15, 16, 17, 21, 22, 23, 24, 26, 28, 29, 31, 33, 34, 35, 39]
Alice's sifted key: [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0]
Bob's sifted key:   [1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Errors in sifted key: 7/24


In [35]:
# Attack Detection
# number of bits to sample from the sifted key
sample_size = int(len(matching_indices) * SAMPLE_FRACTION)
# take first few bits as sample
sample_indices = list(range(sample_size))
# count how many bits do not match between alice and bob
mismatches = sum(1 for i in sample_indices if alice_key[i] != bob_key[i])
error_rate = mismatches / len(sample_indices)

# Final key selection
final_key_indices = [i for i in range(len(alice_key)) if i not in sample_indices]
final_alice_key = [alice_key[i] for i in final_key_indices]
final_bob_key   = [bob_key[i]   for i in final_key_indices]

# Results
print(f"Sample size:  {len(sample_indices)} bits")
print(f"Mismatches:   {mismatches}")
print(f"Error rate:   {error_rate:.2%}  (threshold: {ATTACK_THRESHOLD:.0%})")
print()

# decide whether an attack is happening
if error_rate > ATTACK_THRESHOLD:
    print("Attack detected, aborting key")
else:
    print("No attack detected.")

print("\nFinal key")
print(f"Alice's final key: {final_alice_key}")
print(f"Bob's final key:   {final_bob_key}")
print(f"Keys match: {final_alice_key == final_bob_key}")

Sample size:  12 bits
Mismatches:   4
Error rate:   33.33%  (threshold: 10%)

Attack detected, aborting key

Final key
Alice's final key: [1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0]
Bob's final key:   [1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Keys match: False


In [36]:
expected_error_rate = 0.25 * EVE_INTERCEPT
print(f"Expected error rate: {expected_error_rate:.2%}")
print(f"Observed error rate: {errors_in_sifted / len(alice_key):.2%}")

Expected error rate: 25.00%
Observed error rate: 29.17%
